# 02A Full Refresh Demo

Run the completed retail example: **Full Read → PySpark transformation → Overwrite**. This notebook preserves the working Orders, Products, and Order History pipeline that previously lived in `02_pipeline.ipynb`.

## Tested with FabricOps

This completed demo preserves the Microsoft Fabric runtime validation record of the full-refresh pipeline it was copied from.

| FabricOps release | Tested by | Date tested |
|---|---|---|
| v0.2.5 | Voyce | 12 Oct 2026 |

The Testing/Live execution-plan redesign has local structural validation only; the historical Fabric records above describe the prior scaffold.


# 0. Environment

Run the shared Fabric configuration and import the public APIs used by this pipeline.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import orchestrate_read, orchestrate_write, widget_select_data_contract

# 1. Pipeline Data Contracts

Configure the actual sources and targets before selecting contracts. The widget defaults to **Testing** (red): no data or metadata persistence. Select a frozen contract independently for each participant, or explicitly choose **No Contract** for first-time bootstrap when no active contract exists. Expand the inspector to review every saved rule.

Switch the whole pipeline to **Live** (green) only when publication and metadata commits are intended. Live requires the active frozen version where one exists. Environment names only route work. Changing mode or selections requires building a new plan and rerunning Read → Transform → Write.


In [ ]:
# Explicit participating identities are required even before Lineage exists.
PIPELINE_SOURCES = [{'store': 'Bronze', 'schema': 'demo', 'table_name': 'orders'},
 {'store': 'Bronze', 'schema': 'demo', 'table_name': 'products'},
 {'store': 'Gold', 'schema': 'demo', 'table_name': 'order_history'}]
PIPELINE_TARGETS = [{'store': 'Silver', 'schema': 'demo', 'table_name': 'curated_orders'},
 {'store': 'Gold', 'schema': 'demo', 'table_name': 'customer_summary'}]
CONTRACTS = widget_select_data_contract(
    sources=PIPELINE_SOURCES, targets=PIPELINE_TARGETS, spark_session=spark,
)


In [ ]:
# Run after making explicit widget selections. Rebuild for each new run.
EXECUTION_PLAN = CONTRACTS["build_plan"]()


# 2. Read

`orchestrate_read()` is the plug-and-play governed source read. You provide the source location and read behaviour; FabricOps runs the standard source-side lifecycle underneath and returns the DataFrame plus the results of each check.

**Under the hood**

- **Read** — read the governed source.
- **Freshness** — check the source freshness expectation.
- **Schema** — validate the observed schema against the Data Contract.
- **Grain** — evaluate the configured Grain Guardrail.
- **Data Quality** — evaluate configured DQ rules.
- **Profile** — profile the full source for governance and inspection.

**Parameters**

- The `sources` dictionary key — notebook-facing label, not an orchestrator argument.
- `store` — configured FabricStore key from `00_env_config` such as `Bronze`, `Silver`, or `Gold`.
- `schema` — physical Lakehouse or Warehouse schema containing the source table.
- `table_name` — physical source table name.
- `read_mode` — `full` for the complete source or `incremental` for target-aware incremental processing.
- `query` — optional read-only Warehouse SQL pushdown. Use `None` for a normal table read.
- `read_parameters` — Incremental options such as `watermark_column`; not needed for these Full reads.
- `target_table_id` — required for Incremental reads so FabricOps can resolve target-aware state.
- `grain_discovery_enabled`, `grain_discovery_max_columns`, `grain_discovery_max_candidates` — control candidate Grain profiling, not contract Grain checks.
- `frequency_enabled`, `frequency_top_n` — control Frequency profiling.
- `dq_diagnostics` — `"sample"` (default), `"none"` (metrics only), or `"full"` (lazy failure rows).
- `verbose` — enable stage and final summaries.
- `spark_session` — active Fabric Spark session.

The returned object includes `dataframe`, `table_id`, source check results, profiling results, and orchestration stage metadata. Optional inspection lines are kept commented out so normal runs stay clean.

Clone a Read block for each governed source.

In [ ]:
# Dictionary used to keep source results for transformation and lineage.
sources = {}

## READ 1 — Orders

In [ ]:
source = orchestrate_read(
    execution_plan=EXECUTION_PLAN,
    store="Bronze",
    schema="demo",
    table_name="orders",
    read_mode="full",
    query=None,
    grain_discovery_enabled=True,  # False skips candidate profiling, not Grain Guardrails.
    grain_discovery_max_columns=3,
    grain_discovery_max_candidates=8,
    frequency_enabled=True,  # Set False to skip frequency distributions.
    frequency_top_n=5,
    spark_session=spark,
    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
sources["orders"] = source

## READ 2 — Products

In [ ]:
source = orchestrate_read(
    execution_plan=EXECUTION_PLAN,
    store="Bronze",
    schema="demo",
    table_name="products",
    read_mode="full",
    query=None,
    grain_discovery_enabled=True,  # False skips candidate profiling, not Grain Guardrails.
    grain_discovery_max_columns=3,
    grain_discovery_max_candidates=8,
    frequency_enabled=True,  # Set False to skip frequency distributions.
    frequency_top_n=5,
    spark_session=spark,
    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
sources["products"] = source

## READ 3 — Order History

In [ ]:
source = orchestrate_read(
    execution_plan=EXECUTION_PLAN,
    store="Gold",
    schema="demo",
    table_name="order_history",
    read_mode="full",
    query="""
SELECT
    historical_order_id,
    customer_id,
    order_datetime,
    net_amount
FROM demo.order_history
""",
    grain_discovery_enabled=True,  # False skips candidate profiling, not Grain Guardrails.
    grain_discovery_max_columns=3,
    grain_discovery_max_candidates=8,
    frequency_enabled=True,  # Set False to skip frequency distributions.
    frequency_top_n=5,
    spark_session=spark,
    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
sources["history"] = source

??? info "Optional Read inspection"

    Inspect a governed source result. Select a source key when the notebook reads multiple tables.

In [ ]:
# Inspect the first source result when available.
if sources:
    inspect_source = next(iter(sources))
    display(sources[inspect_source]["dataframe"].limit(10))
else:
    print("No Read results yet. Run the Read cell(s) first.")

# Optional Read result inspection:
# display(sources[inspect_source].get("scope"))  # Incremental reads only.
# display(sources[inspect_source].get("orchestration_stages"))
# display(sources[inspect_source].get("freshness_result"))
# display(sources[inspect_source].get("schema_result"))
# display(sources[inspect_source].get("grain_result"))
# display(sources[inspect_source].get("dq_result"))
# display(sources[inspect_source].get("profile_result"))

# 3. PySpark Transform

Use the named source DataFrames to build the DataFrames you want to publish. This example produces two different outputs so the WRITE blocks demonstrate publishing different tables.

In [ ]:
# Pull the source DataFrames back out of the sources dictionary by name.
orders_df = sources["orders"]["dataframe"]
products_df = sources["products"]["dataframe"]
history_df = sources["history"]["dataframe"]

history_summary_df = (
    history_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("historical_order_count"),
        F.sum("net_amount").alias("historical_net_amount"),
        F.max("order_datetime").alias("latest_historical_order_datetime"),
    )
)

# Output 1: detailed order-level table.
transformed_df = (
    orders_df.alias("orders")
    .join(products_df.alias("products"), on="product_id", how="left")
    .join(history_summary_df.alias("history"), on="customer_id", how="left")
    .withColumn("order_net_amount", F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2))
    .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
    .select(
        "order_id", "customer_id", "order_datetime", "modified_datetime",
        "product_id", "product_name", "product_category", "quantity", "unit_price",
        "discount", "order_net_amount", "order_status", "shipping_country",
        "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
    )
)

# Output 2: customer-level summary derived from the detailed order output.
customer_summary_df = (
    transformed_df
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("order_net_amount").alias("total_order_net_amount"),
        F.max("order_datetime").alias("latest_order_datetime"),
        F.max("historical_order_count").alias("historical_order_count"),
        F.max("historical_net_amount").alias("historical_net_amount"),
    )
)

# Optional development inspection — uncomment only when needed.
# Detailed order-level transformed output that will feed WRITE 1.
# display(transformed_df)
# Customer-level aggregated output that will feed WRITE 2.
# display(customer_summary_df)

# 4. Write

`orchestrate_write()` is the plug-and-play governed target write. You pass the transformed DataFrame, its governed sources, destination, and load strategy; FabricOps resolves the target contract, runs target-side guardrails, and only publishes when the selected contract mode allows it.

**Under the hood**

- **Schema** — validate the target schema against the Data Contract.
- **Sensitive Data** — detect and apply configured sensitive-data treatment.
- **Source Drift** — check governed source changes against target expectations.
- **Grain** — evaluate the configured target Grain Guardrail.
- **Data Quality** — evaluate configured DQ rules.
- **Write** — publish the target when running in Enforce mode.
- **Profile** — profile the persisted target after a successful write.

In **Validate** mode, the same guardrails run but FabricOps stops before `Write`, records the validation outcome, and leaves the target unchanged. In **Enforce** mode, successful checks continue through publication and persisted-target profiling.

**Parameters**

- first argument — transformed Spark DataFrame to validate and publish.
- The `writes` dictionary key — notebook-facing label, not an orchestrator argument.
- `sources` — the `orchestrate_read()` result objects that contributed to this target; their `table_id` values drive lineage and source-side contract checks.
- `store` — configured destination FabricStore key from `00_env_config`.
- `schema` — physical target schema.
- `table_name` — physical target table name.
- `write_mode` — governed write strategy such as `overwrite`, `append`, `scd1`, or `scd2`.
- `write_parameters` — mode-specific options, such as SCD keys and effective dates.
- `contracts` — result from `widget_select_data_contract()` that tells FabricOps whether each governed table is in Validate or Enforce mode and which exact contract applies.
- `repartition_by` — optional Spark partition count before the write.
- `grain_discovery_enabled`, `grain_discovery_max_columns`, `grain_discovery_max_candidates` — control candidate Grain profiling, not contract Grain checks.
- `frequency_enabled`, `frequency_top_n` — control Frequency profiling.
- `dq_diagnostics` — `"sample"` (default), `"none"` (metrics only), or `"full"` (lazy failure rows).
- `existing_mapping` — optional caller-managed Sensitive Data token mapping; omit when a managed mapping store is configured.
- `verbose` — enable stage and final summaries.
- `spark_session` — active Fabric Spark session.

The returned object includes the target `table_id`, guardrail results, publication status, orchestration stage metadata, and, when published, persisted-target profiling results. Optional inspection lines are kept commented out so normal runs stay clean.

Clone a Write block for each governed target.

In [ ]:
# Dictionary used to keep publication results for later reference.
writes = {}

## WRITE 1 — Lakehouse Curated Orders

In [ ]:
write_result = orchestrate_write(
    transformed_df,
    sources=[sources["orders"], sources["products"], sources["history"]],
    store="Silver",
    schema="demo",
    table_name="curated_orders",
    write_mode="overwrite",
    execution_plan=EXECUTION_PLAN,
    repartition_by=None,
    grain_discovery_enabled=True,  # False skips candidate profiling, not Grain Guardrails.
    grain_discovery_max_columns=3,
    grain_discovery_max_candidates=8,
    frequency_enabled=True,  # Set False to skip frequency distributions.
    frequency_top_n=5,
    spark_session=spark,
    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
if not write_result["published"]:
    notebookutils.notebook.exit("Data Contract validation passed for curated_orders_lakehouse; target not published.")
writes["curated_orders_lakehouse"] = write_result

## WRITE 2 — Warehouse Customer Summary

In [ ]:
write_result = orchestrate_write(
    customer_summary_df,
    sources=[sources["orders"], sources["products"], sources["history"]],
    store="Gold",
    schema="demo",
    table_name="customer_summary",
    write_mode="overwrite",
    execution_plan=EXECUTION_PLAN,
    repartition_by=4,
    grain_discovery_enabled=True,  # False skips candidate profiling, not Grain Guardrails.
    grain_discovery_max_columns=3,
    grain_discovery_max_candidates=8,
    frequency_enabled=True,  # Set False to skip frequency distributions.
    frequency_top_n=5,
    spark_session=spark,
    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
if not write_result["published"]:
    notebookutils.notebook.exit("Data Contract validation passed for customer_summary_warehouse; target not published.")
writes["customer_summary_warehouse"] = write_result

# 5. Inspect

Inspect the governed Write result using the same pattern for Full Refresh, Incremental Append, SCD1, and SCD2. Select a target key when multiple targets were written.

The DataFrame preview shows the Write input, not necessarily the persisted target. The persisted target profile is available separately.

??? info "Optional Write inspection"

    Uncomment any result below to inspect the governed target, validation checks, or execution stages.

In [ ]:
# Inspect the first target result when available.
if writes:
    inspect_write = next(iter(writes))
    display(writes[inspect_write]["dataframe"].limit(10))
else:
    print("No Write results yet. Run the Write cell(s) first.")

# Optional Write result inspection:
# display((writes[inspect_write].get("profile_result") or {}).get("profile"))  # Persisted target profile when available.
# display((writes[inspect_write].get("profile_result") or {}).get("frequency_profile"))
# display(writes[inspect_write].get("schema_result"))
# display(writes[inspect_write].get("grain_result"))
# display(writes[inspect_write].get("sensitive_result"))
# display(writes[inspect_write].get("source_drift_results"))
# display(writes[inspect_write].get("dq_result"))
# display(writes[inspect_write].get("orchestration_stages"))
# Inspect the actual mode and outcome; Testing is not successful publication.
# print(writes[inspect_write]["execution_mode"], writes[inspect_write]["publication_status"])

# Refresh expandable Data Contract sections with this plan's execution results.
CONTRACTS["refresh"]()
